# Mushroom Identification Benchmark — Colab Runner

This notebook runs the full comparative benchmark (CNN + Tree + DB + LLM + Unified) on Google Colab.

**Before you start:**
1. Upload `mushroom-benchmark.zip` directly to Colab (drag-and-drop onto the file browser on the left, or use the upload cell below)
2. Runtime → Change runtime type → Select **T4 GPU** (optional but recommended)

**Estimated time:** ~2–4 hours for 57 specimens (with GPU ~1–2 hours)


## Step 1: Upload Project ZIP (if not already uploaded)

If you haven't already dragged `mushroom-benchmark.zip` into the file browser, run this cell to upload it.

In [ ]:
from google.colab import files
import os

zip_path = "/content/mushroom-benchmark.zip"

if not os.path.exists(zip_path):
    print("Upload your mushroom-benchmark.zip file:")
    uploaded = files.upload()
    # Rename to expected name if needed
    for name in uploaded.keys():
        if name.endswith('.zip'):
            os.rename(name, zip_path)
            print(f"Uploaded: {zip_path}")
            break
else:
    print(f"ZIP already present: {zip_path}")

## Step 2: Extract Project

In [ ]:
import shutil
from pathlib import Path

zip_path = Path('/content/mushroom-benchmark.zip')
project_dir = Path('/content/mushroom-project')

if not zip_path.exists():
    raise FileNotFoundError(f"ZIP not found at {zip_path}. Upload it first.")

# Clean up previous extraction if re-running
if project_dir.exists():
    shutil.rmtree(project_dir)

print("Extracting...")
shutil.unpack_archive(str(zip_path), str(project_dir))
print(f"Project extracted to {project_dir}")

# Verify key files
key_files = [
    'benchmarks/evaluation_manifest.csv',
    'data/Yolov8/best.pt',
    'artifacts/cnn_weights.pt',
    'data/raw/key.xml',
    'data/raw/species_traits.xml',
]
missing = [f for f in key_files if not (project_dir / f).exists()]
if missing:
    print(f"\n⚠ Missing: {missing}")
else:
    print("\n✓ All key files present.")

## Step 3: Install Python Dependencies

In [ ]:
%cd /content/mushroom-project

# Core dependencies (skip tensorflow to avoid conflicts with torch)
!pip install -q \
    pandas numpy scikit-learn \
    torch torchvision timm \
    Pillow opencv-python scikit-image \
    h5py tqdm \
    fastapi uvicorn python-multipart \
    pytest pytest-cov \
    python-dotenv pyyaml matplotlib seaborn \
    ultralytics

## Step 4: Verify GPU (Optional but Recommended)

In [ ]:
!nvidia-smi

## Step 5: Install Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## Step 6: Start Ollama Server & Pull Model

In [ ]:
import subprocess
import time
import os

# Kill any existing ollama processes
!pkill -f "ollama serve" 2>/dev/null || true
time.sleep(2)

# Start Ollama in background
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=env
)

# Wait for server to be ready
print("Starting Ollama server...")
for i in range(30):
    time.sleep(1)
    check = subprocess.run(
        ["curl", "-s", "http://localhost:11434"],
        capture_output=True
    )
    if check.returncode == 0:
        print("Ollama server is ready.")
        break
else:
    print("WARNING: Ollama may not be ready yet.")

## Step 7: Pull the Model

In [ ]:
!ollama pull gemma3:4b

## Step 8: Verify Setup

In [ ]:
# Check Ollama is responding and model is loaded
!curl -s http://localhost:11434/api/tags | python3 -m json.tool

# Check project structure
!ls -la benchmarks/evaluation_manifest.csv data/Yolov8/best.pt artifacts/cnn_weights.pt

## Step 9: Keep Colab Alive (Run in Browser Console)

To prevent Colab from disconnecting during the long benchmark run, open your browser's **Developer Tools** (F12), go to the **Console** tab, and paste this code:

```javascript
function ConnectButton(){
    console.log("Keeping Colab alive...");
    document.querySelector("colab-connect-button").click();
}
setInterval(ConnectButton, 60000);
```

This clicks the connect button every 60 seconds.

## Step 10: Run the Benchmark

In [ ]:
import os

os.environ["OLLAMA_TIMEOUT"] = "600"
os.environ["OLLAMA_NUM_PREDICT"] = "512"
os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434"

output_dir = "artifacts/benchmarks/colab_run"
os.makedirs(output_dir, exist_ok=True)

!python3 -m benchmarks.run_comparative \
    --manifest benchmarks/evaluation_manifest.csv \
    --output-dir {output_dir} \
    --methods all

## Step 11: Download Results

In [ ]:
from google.colab import files
import shutil

results_zip = "/content/benchmark_results"
shutil.make_archive(results_zip, 'zip', output_dir)

print("Downloading results...")
files.download(f"{results_zip}.zip")
print("Done!")

## Troubleshooting

**Ollama connection refused**
- Make sure Step 6 ran successfully
- Try: `!ollama serve &` instead of the subprocess approach

**Out of disk space**
- `gemma3:4b` is ~3.3 GB. Check: `!df -h`
- Clear pip cache: `!pip cache purge`

**CUDA/GPU not used by Ollama**
- Ollama auto-detects GPU. If it falls back to CPU, inference will be slower (~same as your laptop)
- Check GPU usage during run: `!nvidia-smi -l 5` in a separate cell

**Session disconnected mid-run**
- Results in `{output_dir}` are lost on disconnect
- Consider running in smaller batches and copying to Drive periodically